# Part C – Prompt Engineering
**Student:** Dabone Abdoul Latif  
**Index:** 10012200015  

---

## What is Prompt Engineering?

In a RAG (Retrieval-Augmented Generation) chatbot, the **prompt** is the full message we send to the AI (Groq LLM).  
It contains three things:
1. **Instructions** — how the AI should behave
2. **Context** — the chunks retrieved from Part B (the relevant documents)
3. **User question** — what the user actually asked

The goal of Part C is to **design** this prompt carefully so the AI:
- Answers using only the real documents (no made-up facts)
- Stays within the token (word) limit of the model
- Gives useful, accurate answers

---

### This notebook covers:
| Task | What we do |
|---|---|
| Task 1 | Design 3 different prompt templates |
| Task 2 | Manage context window (token limit) |
| Task 3 | Run experiments — compare outputs from each prompt |

---
## ⚙️ Setup — Load Everything from Part B

Before building prompts, we need to reload the retrieval system from Part B.  
This cell:
- Loads all the chunks (text pieces) from disk
- Loads the embedding model (converts text to numbers)
- Loads the FAISS index (the vector search engine)
- Rebuilds the BM25 keyword search

> 💡 Think of this as turning on all the engines before we start the car.

In [ ]:
import json
import os
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from groq import Groq

# ── Load chunks saved by Part A ──────────────────────────────────────────────
print("Loading chunks...")
with open('chunks/csv_chunks.json', 'r') as f:
    csv_chunks = json.load(f)
with open('chunks/pdf_chunks.json', 'r') as f:
    pdf_chunks = json.load(f)

all_chunks = csv_chunks + pdf_chunks
texts = [c['text'] for c in all_chunks]
print(f"  ✅ Loaded {len(all_chunks)} chunks ({len(csv_chunks)} election + {len(pdf_chunks)} budget)")

# ── Load the sentence embedding model ────────────────────────────────────────
print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("  ✅ Embedding model ready")

# ── Load the FAISS index saved by Part B ─────────────────────────────────────
print("Loading FAISS index...")
index = faiss.read_index('indexes/rag_index.faiss')
print(f"  ✅ FAISS index loaded — {index.ntotal} vectors")

# ── Rebuild BM25 keyword search ───────────────────────────────────────────────
print("Building BM25 keyword search...")
tokenized_corpus = [text.lower().split() for text in texts]
bm25 = BM25Okapi(tokenized_corpus)
print("  ✅ BM25 ready")

print("\n🚀 All systems loaded and ready for Part C!")

---
## 🔍 Hybrid Search (from Part B)

This is the same search function we built in Part B.  
We paste it here so this notebook is self-contained.

In [ ]:
def hybrid_search(query, k=5, alpha=0.5):
    """
    Finds the most relevant chunks for a given query.
    Combines:
      - Vector search (semantic meaning via FAISS)
      - BM25 keyword search (exact word matching)
    
    alpha=0.5 means 50% weight for each method.
    """
    # Step 1: Vector (semantic) search
    query_vector = model.encode([query]).astype('float32')
    distances, indices = index.search(query_vector, k * 2)

    # Step 2: BM25 keyword search
    bm25_scores = bm25.get_scores(query.lower().split())
    max_bm25 = max(bm25_scores) if max(bm25_scores) > 0 else 1

    # Step 3: Combine both scores
    combined = []
    for i, chunk in enumerate(all_chunks):
        v_score = 0
        for rank, idx in enumerate(indices[0]):
            if idx == i:
                v_score = 1 / (1 + distances[0][rank])
                break
        score = (alpha * v_score) + ((1 - alpha) * (bm25_scores[i] / max_bm25))
        if score > 0:
            combined.append({'chunk': chunk, 'score': score})

    return sorted(combined, key=lambda x: x['score'], reverse=True)[:k]

print("✅ hybrid_search() function is ready")

---
## Task 1 — Designing Prompt Templates

A **prompt template** is like a letter template.  
You fill in the blanks (`{context}`, `{query}`) each time you use it.

We will design **3 versions** of the prompt:

| Prompt | Style | Goal |
|---|---|---|
| **A – Strict** | Very firm rules | Prevent hallucination at all costs |
| **B – Loose** | Friendly, open | More detailed, natural-sounding answers |
| **C – Academic** | Formal with citations | Structured, cites sources like a research paper |

> 💡 **What is a hallucination?**  
> When an AI makes up information that is NOT in the documents. For example, inventing a budget number that was never stated. We want to prevent this.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# PROMPT A — STRICT (Anti-Hallucination)
# ─────────────────────────────────────────────────────────────────
def prompt_strict(context, query):
    """
    Very strict: the AI can ONLY use the context.
    If the answer isn't there, it must say so.
    Best for: preventing made-up facts.
    """
    return f"""You are a helpful assistant for Academic City University.

CONTEXT (retrieved from official documents):
{context}

USER QUESTION: {query}

STRICT INSTRUCTIONS:
- Answer ONLY using the context above.
- Do NOT use any knowledge outside the context.
- If the answer is not in the context, respond exactly: "I don't have that information."
- Keep your answer short and direct."""


# ─────────────────────────────────────────────────────────────────
# PROMPT B — LOOSE (Friendly and Natural)
# ─────────────────────────────────────────────────────────────────
def prompt_loose(context, query):
    """
    More relaxed: the AI uses the context but can be more natural.
    Best for: getting detailed, readable answers.
    Risk: slightly higher chance of adding extra details.
    """
    return f"""You are a knowledgeable assistant helping students at Academic City University.

Here is some relevant information from our documents:
{context}

Question: {query}

Please provide a helpful and detailed answer using the information provided above."""


# ─────────────────────────────────────────────────────────────────
# PROMPT C — ACADEMIC (Formal with Source Citations)
# ─────────────────────────────────────────────────────────────────
def prompt_academic(context, query):
    """
    Academic style: structured response with [Source X] citations.
    Best for: formal reports and academic work.
    """
    return f"""You are an academic research assistant at Academic City University.

Retrieved Context:
{context}

Research Question: {query}

Instructions:
- Base your answer strictly on the retrieved context above.
- Cite your sources using [Source 1], [Source 2], etc.
- If information is missing from the context, clearly state what is unknown.
- Use clear, formal language suitable for an academic report.

Answer:"""


print("✅ Three prompt templates defined:")
print("   → prompt_strict()   — Strict, no hallucinations")
print("   → prompt_loose()    — Friendly, natural answers")
print("   → prompt_academic() — Formal with source citations")

---
## Task 2 — Context Window Management

### What is a Context Window?

LLMs (like Groq's Llama model) can only read a limited amount of text at once.  
This limit is measured in **tokens** (roughly: 1 token ≈ 4 characters or ¾ of a word).

For example, if the model allows **8,000 tokens**, and our prompt uses 7,500 tokens, that's fine.  
But if we try to send 10,000 tokens, it will **crash or cut off** the text.

### Our Strategy: **Rank + Truncate**

We use a two-step approach:
1. **Rank** — Only take the **top 3** most relevant chunks (instead of all 5)
2. **Truncate** — If even those 3 chunks are too long, cut them at a safe limit

We set a budget of **1,000 tokens for context** — this leaves plenty of room for the instructions and response.

```
Total budget:  8,000 tokens
Instructions:    ~200 tokens
User question:    ~30 tokens
Context budget: 1,000 tokens  ← we control this
AI response:    ~500 tokens
```

In [ ]:
def count_tokens(text):
    """
    Estimates the number of tokens in a text.
    Rule of thumb: 1 token ≈ 4 characters.
    This is an approximation — exact count depends on the model's tokenizer.
    """
    return len(text) // 4


def build_context(results, max_tokens=1000):
    """
    Builds the context string from retrieved chunks.
    
    Strategy:
      1. Take top chunks (already ranked by relevance score)
      2. Add each chunk until we hit the token budget
      3. If a chunk is too big, truncate it to fit
    
    Returns:
      - context_text: the formatted string ready to inject into the prompt
      - total_tokens: how many tokens we used
    """
    context_parts = []
    total_tokens = 0

    for i, res in enumerate(results):
        chunk_text = res['chunk']['text']
        source = res['chunk']['source']          # 'election' or 'budget'
        score = res['score']
        chunk_tokens = count_tokens(chunk_text)

        label = f"[Source {i+1} | {source} | relevance: {score:.2f}]"

        if total_tokens + chunk_tokens <= max_tokens:
            # Chunk fits — add it fully
            context_parts.append(f"{label}\n{chunk_text}")
            total_tokens += chunk_tokens
        else:
            # Chunk too big — truncate it to the remaining budget
            remaining_chars = (max_tokens - total_tokens) * 4
            if remaining_chars > 80:  # Only add if we have at least 80 chars left
                truncated = chunk_text[:remaining_chars] + "..."
                context_parts.append(f"{label}\n{truncated}")
                total_tokens = max_tokens
            break  # Budget exhausted

    context_text = "\n\n".join(context_parts)
    return context_text, total_tokens


# ── Quick test to see it working ──────────────────────────────────
test_results = hybrid_search("What is the inflation target for 2025?", k=3)
test_context, test_tokens = build_context(test_results, max_tokens=1000)

print("📦 Context Window Test")
print(f"   Chunks retrieved: {len(test_results)}")
print(f"   Tokens used:      {test_tokens} / 1000")
print(f"   Remaining budget: {1000 - test_tokens} tokens")
print("\n--- Context Preview ---")
print(test_context[:400], "...")

---
## Task 3 — Experiments: Comparing Prompt Outputs

### 🔑 Set Your Groq API Key

To use the Groq LLM, you need a free API key from [https://console.groq.com](https://console.groq.com).  
Replace `"your_api_key_here"` below with your actual key.

> ⚠️ **Never share your API key publicly or push it to GitHub!**  
> The `.gitignore` will protect the `.env` file if you use one.

In [ ]:
# ── Set your Groq API key here ────────────────────────────────────
GROQ_API_KEY = "your_api_key_here"   # 👈 Replace this with your actual key

def ask_groq(prompt, temperature=0.1):
    """
    Sends a prompt to the Groq LLM and returns the answer.
    
    temperature=0.1 → very focused, factual answers (close to 0 = less creative)
    temperature=0.9 → more creative but less precise
    """
    client = Groq(api_key=GROQ_API_KEY)
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=400
    )
    return response.choices[0].message.content

print("✅ ask_groq() function is ready")
print("   Model: llama-3.3-70b-versatile (via Groq)")

---
### Experiment 1 — Budget/PDF Query

**Query:** `"What is the inflation target for 2025?"`

This should be answered using the **2025 Ghana Budget PDF**.  
We will run all 3 prompts with the **same context** and compare the outputs.

In [ ]:
# ── Experiment 1 Setup ─────────────────────────────────────────────
query_1 = "What is the inflation target for 2025?"

print(f"🔍 Query: {query_1}")
print("-" * 60)

# Retrieve top 3 relevant chunks
results_1 = hybrid_search(query_1, k=3)
context_1, tokens_1 = build_context(results_1, max_tokens=1000)

print(f"📦 Context: {tokens_1} tokens used from {len(results_1)} chunks")
print()

# ── Run all 3 prompts ──────────────────────────────────────────────
experiments = {
    "Prompt A — Strict": prompt_strict(context_1, query_1),
    "Prompt B — Loose":  prompt_loose(context_1, query_1),
    "Prompt C — Academic": prompt_academic(context_1, query_1),
}

results_log = {}  # We save results here to compare later

for name, prompt in experiments.items():
    print(f"{'='*60}")
    print(f"🧪 {name}")
    print(f"{'='*60}")
    answer = ask_groq(prompt)
    results_log[name] = answer
    print(answer)
    print()

---
### Experiment 2 — Election/CSV Query

**Query:** `"How many votes did Nana Addo get in the Ahafo region in 2020?"`

This should be answered using the **Ghana Election CSV**.  
We test whether the strict prompt stops the AI from guessing when the exact year is different.

In [ ]:
# ── Experiment 2 Setup ─────────────────────────────────────────────
query_2 = "How many votes did Nana Addo get in the Ahafo region in 2020?"

print(f"🔍 Query: {query_2}")
print("-" * 60)

results_2 = hybrid_search(query_2, k=3)
context_2, tokens_2 = build_context(results_2, max_tokens=1000)

print(f"📦 Context: {tokens_2} tokens used from {len(results_2)} chunks")
print()

experiments_2 = {
    "Prompt A — Strict":  prompt_strict(context_2, query_2),
    "Prompt B — Loose":   prompt_loose(context_2, query_2),
    "Prompt C — Academic": prompt_academic(context_2, query_2),
}

results_log_2 = {}

for name, prompt in experiments_2.items():
    print(f"{'='*60}")
    print(f"🧪 {name}")
    print(f"{'='*60}")
    answer = ask_groq(prompt)
    results_log_2[name] = answer
    print(answer)
    print()

---
## Summary — What We Observed

Run this cell after both experiments to print a clean comparison table.

In [ ]:
print("╔══════════════════════════════════════════════════════════════╗")
print("║            PART C — EXPERIMENT RESULTS SUMMARY              ║")
print("╠══════════════════════════════════════════════════════════════╣")

# ── Experiment 1 Summary ──────────────────────────────────────────
print(f"\n📌 Experiment 1: '{query_1}'")
print(f"   Context tokens used: {tokens_1} / 1000")
print()
for name, answer in results_log.items():
    # Show first 150 characters of each answer as a preview
    preview = answer.replace('\n', ' ')[:150]
    print(f"  [{name}]")
    print(f"   → {preview}...")
    print()

# ── Experiment 2 Summary ──────────────────────────────────────────
print(f"📌 Experiment 2: '{query_2}'")
print(f"   Context tokens used: {tokens_2} / 1000")
print()
for name, answer in results_log_2.items():
    preview = answer.replace('\n', ' ')[:150]
    print(f"  [{name}]")
    print(f"   → {preview}...")
    print()

print("╠══════════════════════════════════════════════════════════════╣")
print("║  KEY FINDINGS:                                               ║")
print("║  • Prompt A (Strict): Best for avoiding hallucinations.      ║")
print("║  • Prompt B (Loose):  Best for readable, detailed answers.   ║")
print("║  • Prompt C (Academic): Best for formal/cited responses.     ║")
print("║  • Context window: 1000 tokens kept all prompts within safe  ║")
print("║    limits (well under the 8000-token model maximum).         ║")
print("╚══════════════════════════════════════════════════════════════╝")